# Wildfire-RL — Quickstart Demo

Minimal end-to-end demo using the installed `wildfire_rl` package (no Colab, no hardcoded paths).
Run `pip install -e .` first. This trains a tiny PPO and compares it to random / no-op baselines.

In [ ]:
import numpy as np
from wildfire_rl.config import EnvConfig, PPOConfig
from wildfire_rl.envs.base import make_env_factory
from wildfire_rl.train.ppo import train_ppo
from wildfire_rl.eval.evaluate import evaluate_policy
from wildfire_rl.eval.baselines import RandomPolicy, NoOpPolicy

## 1. Build a small state tensor
Use the committed sample, or synthesize a tiny 7-channel tensor.

In [ ]:
rng = np.random.default_rng(0)
tensor = rng.random((7, 16, 16)).astype('float32')
tensor[0] = 0.0
tensor[0, 8, 8] = 1.0  # one central ignition
env_cfg = EnvConfig(grid_size=16, max_steps=40)
factory = make_env_factory(state_tensor=tensor, config=env_cfg)

## 2. Train a tiny PPO

In [ ]:
ppo_cfg = PPOConfig(total_timesteps=3000, n_steps=256, batch_size=64, features_dim=64)
model = train_ppo(factory, ppo_cfg, seed=0)

## 3. Compare PPO against baselines
A learned policy is only meaningful relative to random / no-op controls.

In [ ]:
action_space = factory().action_space
policies = {
    'ppo': model,
    'random': RandomPolicy(action_space, seed=0),
    'noop': NoOpPolicy(action_space),
}
for name, policy in policies.items():
    summary = evaluate_policy(policy, factory, n_episodes=5, base_seed=0)['summary']
    print(f"{name:7s} reward={summary['episode_reward_mean']:.2f} "
          f"burned={summary['burned_cells_mean']:.1f}")

Next: train for real with `wildfire-rl train --config configs/experiment/multiseed.yaml`,
then `wildfire-rl transfer --config configs/experiment/transfer.yaml`. See the README.